**<font size="7">Análise das emoções de Relatórios Financeiros </font>**


**O Modelo 1 é apenas um teste que eu fiz em um único relatório, <mark>o Modelo 2 é código final.</mark>**

**OBS:** Os dados no dicionário de emoções estão desbalanceados.

# Carregar Bibliotecas

In [ ]:
#pip install -U sentence-transformers

In [ ]:
# OBS: Não consegui instalar no meu pc. Por isso, só consigo rodar este notebook no Google Colab
#pip install tf-keras

In [ ]:
#pip install openpyxl

In [ ]:
from transformers import (
    AutoTokenizer,
    BertForSequenceClassification,
    pipeline,
)
import pandas as pd
import nltk
import re

# Baixar o modelo de segmentação de frases do NLTK
nltk.download('punkt') # Necessário para segmentação de sentenças
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize


from warnings import filterwarnings
filterwarnings('ignore') # remover avisos

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

# Importar dados

In [ ]:
# Tabela com os relatórios
df = pd.read_excel("/content/Relatorio_Planilha_Analise_Sentimento1.xlsx")
print(f"Dimensões:{df.shape}")
df.head(3)

Dimensões:(50, 5)


,Id_Empresa,Nome,t-1,t,t+2
0,1,Petrobras,A Petróleo Brasileiro S.A. Petrobras dedicase...,A Petróleo Brasileiro S.A. - Petrobras dedica-...,"A Petróleo Brasileiro S.A. – Petrobras, Socied..."
1,2,Ambev,A Companhia de Bebidas das Américas – Ambev (r...,\nA Ambev S.A. (referida como “Companhia” ou “...,"A Ambev S.A. (referida como “Companhia”, “Ambe..."
2,3,Via S.A,"\n\nA Via Varejo S.A., diretamente ou por meio...","\n\nA Via Varejo S.A., diretamente ou por meio...","A Via S.A., diretamente ou por meio de suas co..."


### Dicionários de emoções
**OBS:** Os dados estão desbalanceados.

In [ ]:
# Dicionários de emoções
df_alegria=  pd.read_excel("dicionario_sentimentos_v3.0.xlsx", sheet_name=1)
df_medo = pd.read_excel("dicionario_sentimentos_v3.0.xlsx", sheet_name=2)
df_raiva = pd.read_excel("dicionario_sentimentos_v3.0.xlsx", sheet_name=3)
df_repugnancia = pd.read_excel("dicionario_sentimentos_v3.0.xlsx", sheet_name=4)
df_surpresa = pd.read_excel("/content/dicionario_sentimentos_v3.0.xlsx", sheet_name=5)
df_tristeza = pd.read_excel("/content/dicionario_sentimentos_v3.0.xlsx", sheet_name=6)

df_tristeza.head(3)

,Tristeza
0,Abdicar
1,Abaixo dos
2,padrões estabelecidos


In [ ]:
# Dimensões dos dicionários
print(f"Dimensões Alegria:{df_alegria.shape}")
print(f"Dimensões Medo:{df_medo.shape}")
print(f"Dimensões Raiva:{df_raiva.shape}")
print(f"Dimensões Repugnância:{df_repugnancia.shape}")
print(f"Dimensões Surpresa:{df_surpresa.shape}")

Dimensões Alegria:(302, 1)
Dimensões Medo:(158, 1)
Dimensões Raiva:(147, 1)
Dimensões Repugnância:(110, 1)
Dimensões Surpresa:(73, 1)


# Preparar dados

### Preparar dicionário

In [ ]:
# Transformar as emoções em lista
list_alegria = df_alegria['Alegria'].tolist()
list_medo = df_medo['Medo'].tolist()
list_raiva = df_raiva['Raiva'].tolist()
list_repugnancia = df_repugnancia['Repugnância'].tolist()
list_surpresa = df_surpresa['Surpresa'].tolist()
print(list_alegria ) # Exibir a lista "alegria"

['Abastança', 'Abono', 'Abrandar', 'Abrilhantar', 'Abundância', 'Acalmar', 'Acessível', 'Aclamado', 'Acolhedor', 'Adiantamento', 'Admirar', 'Afável', 'Completar', 'Compreensível', 'Conceder herança', 'Conciliação', 'Concluir', 'Concordar', 'Condecoração', 'Confiança', 'Congratulação', 'Conquista', 'Consagrado', 'Constância', 'Evolução', 'Excelência', 'Excepcional', 'Excitação', 'Executar', 'Êxito', 'Êxtase', 'Extraordinário', 'Fácil', 'Fantástico', 'Fartura', 'Favorável', 'Merecedor', 'Meritório', 'Momento de alta', 'Muito', 'Notável', 'Notório', 'Obtenção', 'Ocasião propícia', 'Oportunidade', 'Otimista', 'Ótimo', 'Ousadia', 'Rentável', 'Reparação', 'Representação de uma ideia', 'Reputação', 'Resolução', 'Respeito', 'Resultado positivo', 'Retidão', 'Retificação', 'Retribuição', 'Revogação', 'Revolucionar', 'Afirmativo', 'Afortunado', 'Agradável', 'Ajuda', 'Alcançar', 'Alegre', 'Alertar', 'Aliança', 'Alívio', 'Amável', 'Aperfeiçoar', 'Apoiar', 'Aprazível', 'Apreciável', 'Aprimorar', 'Ap

In [ ]:
# Colocar as listas em um dicionário
dicionario_emocoes = {
    "alegria": list_alegria,
    "medo": list_medo,
    "raiva": list_raiva,
    "repugnância": list_repugnancia,
    "surpresa": list_surpresa,
}
print(dicionario_emocoes)

{'alegria': ['Abastança', 'Abono', 'Abrandar', 'Abrilhantar', 'Abundância', 'Acalmar', 'Acessível', 'Aclamado', 'Acolhedor', 'Adiantamento', 'Admirar', 'Afável', 'Completar', 'Compreensível', 'Conceder herança', 'Conciliação', 'Concluir', 'Concordar', 'Condecoração', 'Confiança', 'Congratulação', 'Conquista', 'Consagrado', 'Constância', 'Evolução', 'Excelência', 'Excepcional', 'Excitação', 'Executar', 'Êxito', 'Êxtase', 'Extraordinário', 'Fácil', 'Fantástico', 'Fartura', 'Favorável', 'Merecedor', 'Meritório', 'Momento de alta', 'Muito', 'Notável', 'Notório', 'Obtenção', 'Ocasião propícia', 'Oportunidade', 'Otimista', 'Ótimo', 'Ousadia', 'Rentável', 'Reparação', 'Representação de uma ideia', 'Reputação', 'Resolução', 'Respeito', 'Resultado positivo', 'Retidão', 'Retificação', 'Retribuição', 'Revogação', 'Revolucionar', 'Afirmativo', 'Afortunado', 'Agradável', 'Ajuda', 'Alcançar', 'Alegre', 'Alertar', 'Aliança', 'Alívio', 'Amável', 'Aperfeiçoar', 'Apoiar', 'Aprazível', 'Apreciável', 'Apr

### Preparar df (relatórios)

In [ ]:
# Limpar valores NaN e garantir que tudo seja string
df['t-1'] = df['t-1'].fillna("").astype(str)
df['t'] = df['t'].fillna("").astype(str)
df['t+2'] = df['t+2'].fillna("").astype(str)

# Funções

## Função: Contagem das categorias

In [ ]:
# Contagem das categorias de cada coluna
def contagem_classes(df, cols):
  ''' Função que faz a contagem de classes de cada coluna
      Args:
        - df: dataframe com as colunas a serem contadas
        - cols: lista com os nomes das colunas
  '''

  # Criar um dicionário vazio para armaezar os dados
  result = {}
  # Loop para : Contagem de valores de cada categoria
  for col in cols:
    result[col]= df[col].value_counts()

  # Tranformar o dicionário em DataFrame e fazer sua Transposição
  df_result = pd.DataFrame(result)#.T
  # Substituir NA por 0. E, converter todas as contagens para 'int64'
  df_result = df_result.fillna(0).astype('int64')
  return df_result

 # Modelo 1: Só com único relatório
 Este é um modelo teste, só rodei com um único relatório para testar. O código final está no modelo 2.  

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Modelo pré-treinado (aceita textos em português)
modelo_sbert = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

# O dicionário é codificado pelo modelo SBERT para gerar seu embedding
embeddings_emocoes = {emocao: modelo_sbert.encode(" ".join(map(str, lista))) for emocao, lista in dicionario_emocoes.items()}
# Para cada emoção no dicionario_emocoes, as palavras associadas a ela são unidas em uma única string
#  com " ".join(map(str, lista)) e, em seguida, o modelo gera o embedding dessa string.

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# Exibir embeddings do dicionário de emoções
for chave, valor in embeddings_emocoes.items():
    print(f"{chave}: {valor[:5]}")  # Exibir os primeiros números de cada lista de valores

alegria: [ 0.08052162  0.2852479   0.11884767  0.03268695 -0.19898008]
medo: [0.12895432 0.1503499  0.05885961 0.13182724 0.0835663 ]
raiva: [ 0.09908537  0.13059825 -0.02357616  0.05435435  0.16301349]
repugnância: [ 0.13294192  0.21273977 -0.02310401  0.06294704  0.0630016 ]
surpresa: [ 0.10175819  0.14125508 -0.1068872   0.04164942  0.07206219]


In [ ]:
# Pegando só um relatório
texto = df['t-1'][0]
texto

'A Petróleo Brasileiro S.A.  Petrobras dedicase, diretamente ou por meio de suas subsidiárias e controladas (denominadas, em conjunto, “Petrobras” ou a “Companhia”), à pesquisa, lavra, refino, processamento, comércio e transporte de petróleo proveniente de poço, de xisto ou de outras rochas, de seus derivados, de gás natural e de outros hidrocarbonetos fluidos, além das atividades vinculadas à energia, podendo promover pesquisa, desenvolvimento, produção, transporte, distribuição e comercialização de todas as formas de energia, bem como quaisquer outras atividades correlatas ou afins. A sede social da Companhia está localizada no Rio de Janeiro  RJ.\nAs demonstrações contábeis incluem:\n As demonstrações contábeis consolidadas estão sendo apresentadas de acordo com os padrões internacionais de demonstrações contábeis (IFRS) emitidos pelo International Accounting Standards Board  IASB e também de acordo com práticas contábeis adotadas no Brasil.\n As demonstrações contábeis individuais 

In [ ]:
# O texto do relatório é codificado pelo modelo SBERT para gerar seu embedding.
embedding_texto = modelo_sbert.encode(texto)
embedding_texto[:30] # Exibir os primeiros valores

array([ 0.01755016, -0.13767509, -0.03087044, -0.03717567, -0.0446121 ,
       -0.23386687,  0.00959395,  0.27701497,  0.13821588, -0.07641162,
        0.00738324,  0.09215353, -0.19796437, -0.03552497, -0.23027293,
       -0.20190232,  0.07182094, -0.27487418,  0.32575488, -0.01352627,
        0.45432383, -0.04644673, -0.16989584, -0.01763631, -0.38970712,
        0.3518448 , -0.15578893,  0.02300895,  0.12211439, -0.289873  ],
      dtype=float32)

In [ ]:
# Calcular similaridade entre o texto e cada emoção
similaridades = {
    emocao: cosine_similarity([embedding_texto], [embedding])[0][0]
    for emocao, embedding in embeddings_emocoes.items()
}

# Exibir resultados ordenados
df_similaridades = pd.DataFrame(similaridades.items(), columns=['Emoção', 'Similaridade'])
df_similaridades = df_similaridades.sort_values(by='Similaridade', ascending=False)
print(df_similaridades)

        Emoção  Similaridade
2        raiva      0.168656
3  repugnância      0.136497
1         medo      0.105753
0      alegria      0.070319
4     surpresa      0.029384


# Modelo 2: Usando toda a tabela

Esse é o código final

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Modelo pré-treinado (aceita textos em português)
modelo_sbert = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

# O dicionário é codificado pelo modelo SBERT para gerar seu embedding
embeddings_emocoes = {emocao: modelo_sbert.encode(" ".join(map(str, lista))) for emocao, lista in dicionario_emocoes.items()}
# Para cada emoção no dicionario_emocoes, as palavras associadas a ela são unidas em uma única string
#  com " ".join(map(str, lista)) e, em seguida, o modelo gera o embedding dessa string.

In [ ]:
def calcular_similaridades(linha, t):
    texto = str(linha[t]) if pd.notna(linha[t]) else ""
    # O texto do relatório é codificado pelo modelo SBERT para gerar seu embedding.
    embedding_texto = modelo_sbert.encode(texto)

    # Calcular as similaridades
    similaridades = {
        emocao: cosine_similarity([embedding_texto], [embedding])[0][0]
        for emocao, embedding in embeddings_emocoes.items()
    }


    # Determinar o sentimento predominante
    sentimentos = {key: value for key, value in similaridades.items()}
    similaridades[f'Sentimento_{t}'] = max(sentimentos, key=sentimentos.get)  # Determinar a emoção predominante

    return similaridades

# Aplicar a função em todas as linhas
df_tf_t1 = pd.DataFrame(df.apply(lambda linha: calcular_similaridades(linha, t='t-1'), axis=1).tolist())
df_tf_t = pd.DataFrame(df.apply(lambda linha: calcular_similaridades(linha, t='t'), axis=1).tolist())
df_tf_t2 = pd.DataFrame(df.apply(lambda linha: calcular_similaridades(linha, t='t+2'), axis=1).tolist())

# Concatenar as tabelas
df_resultados_tf1 = pd.concat([df_tf_t1[['Sentimento_t-1']], df_tf_t[['Sentimento_t']], df_tf_t2[['Sentimento_t+2']]], axis=1)
# Exibir os resultados
df_resultados_tf1.head()

,Sentimento_t-1,Sentimento_t,Sentimento_t+2
0,raiva,raiva,raiva
1,raiva,raiva,raiva
2,repugnância,repugnância,raiva
3,repugnância,repugnância,raiva
4,raiva,raiva,raiva


In [ ]:
# Aplicar a função para calcular a contagem
cols = ['Sentimento_t-1','Sentimento_t','Sentimento_t+2']
df_cont1 = contagem_classes(df_resultados_tf1, cols)
df_cont1

,Sentimento_t-1,Sentimento_t,Sentimento_t+2
medo,9,7,9
raiva,19,18,16
repugnância,22,25,24
surpresa,0,0,1
